In [1]:
import pandas as pd
from pathlib import Path

# Load the newly generated metrics
BASE_DIR = Path.cwd().parent 
df_metrics = pd.read_csv(BASE_DIR / "data" / "processed" / "calculated_performance_metrics.csv")

# 1. Null Check
print("Missing Values:\n", df_metrics.isnull().sum())

# 2. Boundary Checks
print("\n--- Diagnostic Boundaries ---")
print(f"Max Volatility (Should be > 0): {df_metrics['Calculated_Volatility'].max():.4f}")
print(f"Min Volatility (Should be > 0): {df_metrics['Calculated_Volatility'].min():.4f}")
print(f"Max Sharpe (Usually < 3.0): {df_metrics['Calculated_Sharpe'].max():.2f}")
print(f"Min Max_Drawdown (Should be negative): {df_metrics['Calculated_Max_Drawdown'].max():.2f}")

# 3. View the Top 5 by Sharpe to ensure it looks reasonable
print("\nTop 5 Funds by Sharpe Ratio:")
display(df_metrics.sort_values('Calculated_Sharpe', ascending=False).head(5))

Missing Values:
 amfi_code                  0
Calculated_Ann_Return      0
Calculated_Volatility      0
Calculated_Sharpe          0
Calculated_Max_Drawdown    0
dtype: int64

--- Diagnostic Boundaries ---
Max Volatility (Should be > 0): 0.2580
Min Volatility (Should be > 0): 0.0049
Max Sharpe (Usually < 3.0): 1.64
Min Max_Drawdown (Should be negative): -0.00

Top 5 Funds by Sharpe Ratio:


,amfi_code,Calculated_Ann_Return,Calculated_Volatility,Calculated_Sharpe,Calculated_Max_Drawdown
34,148567,0.297414,0.141937,1.637441,-0.112657
30,120843,0.296776,0.158870,1.458908,-0.129740
36,148569,0.306736,0.176740,1.367749,-0.163967
19,119551,0.247966,0.137414,1.331491,-0.150124
25,120505,0.315124,0.192909,1.296589,-0.181885


In [3]:
import sqlite3
import pandas as pd
from pathlib import Path

def test_sql_updates():
    print("Initiating SQL Verification Test...")
    
    # FIX: Added .parent so the notebook steps out into the main project folder
    BASE_DIR = Path.cwd().parent 
    db_path = BASE_DIR / "data" / "db" / "bluestock_mf.db"
    
    try:
        conn = sqlite3.connect(db_path)
        # We need to explicitly check if the database actually has tables to ensure it's not a newly created empty file
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        if not cursor.fetchall():
            raise ValueError("Database connected, but it is empty. Path might still be slightly off.")
        print("Successfully connected to the populated database.")
    except Exception as e:
        print(f"Database connection failed: {e}")
        return

    # Test Query 10 (AMC Market Share) to verify the new alias
    test_query = """
    SELECT fund_house, SUM(aum_crore) AS total_amc_aum_crore
    FROM fact_performance
    GROUP BY fund_house
    ORDER BY total_amc_aum_crore DESC
    LIMIT 3;
    """
    
    try:
        print("\nExecuting Query 10...")
        df_result = pd.read_sql(test_query, conn)
        
        print("\n--- Query Output ---")
        print(df_result.to_string(index=False))
        
        print("\n--- Column Verification ---")
        print(df_result.columns.tolist())
        
    except Exception as e:
        print(f"SQL Execution Failed: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    test_sql_updates()

Initiating SQL Verification Test...
Successfully connected to the populated database.

Executing Query 10...

--- Query Output ---
         fund_house  total_amc_aum_crore
    Nippon India MF             154328.0
  Kotak Mahindra MF             145689.0
ICICI Prudential MF             120241.0

--- Column Verification ---
['fund_house', 'total_amc_aum_crore']


In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

# 1. Resolve Path to Database
BASE_DIR = Path.cwd().parent # Adjust if running from a .py script instead of notebook
DB_PATH = BASE_DIR / "data" / "db" / "bluestock_mf.db"

def run_schema_diagnostic():
    if not DB_PATH.exists():
        print(f"❌ ERROR: Database file not found at {DB_PATH}")
        return

    conn = sqlite3.connect(DB_PATH)
    
    print("=== 🏗️ DATABASE SCHEMA DIAGNOSTIC ===\n")
    
    # 2. Check if all 5 tables exist
    tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
    tables = pd.read_sql(tables_query, conn)['name'].tolist()
    
    expected_tables = ['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance']
    missing_tables = [t for t in expected_tables if t not in tables]
    
    if missing_tables:
        print(f"❌ FAILED: Missing tables: {missing_tables}")
    else:
        print(f"✅ PASSED: All {len(expected_tables)} required tables exist.\n")

    # 3. Deep Dive: Verify fact_nav columns (Checking for 'nav_date' vs 'date')
    print("--- Checking fact_nav structure ---")
    nav_columns = pd.read_sql("PRAGMA table_info(fact_nav);", conn)
    display(nav_columns[['name', 'type', 'notnull', 'pk']])
    
    if 'nav_date' in nav_columns['name'].values and 'daily_return' in nav_columns['name'].values:
        print("✅ PASSED: fact_nav has the correct specific columns (nav_date, daily_return).")
    else:
        print("❌ FAILED: fact_nav has incorrect column names. Schema was overwritten.")

    # 4. Check for Foreign Keys
    print("\n--- Checking Foreign Key Constraints ---")
    fk_check = pd.read_sql("PRAGMA foreign_key_list(fact_transactions);", conn)
    if not fk_check.empty:
        print(f"✅ PASSED: Found {len(fk_check)} foreign key links in fact_transactions.")
    else:
        print("❌ FAILED: No foreign keys found. Schema integrity is compromised.")

    conn.close()

# Run the diagnostic
run_schema_diagnostic()

=== 🏗️ DATABASE SCHEMA DIAGNOSTIC ===

❌ FAILED: Missing tables: ['dim_date']
--- Checking fact_nav structure ---


,name,type,notnull,pk
0,amfi_code,INTEGER,0,0
1,date,TEXT,0,0
2,nav,REAL,0,0


❌ FAILED: fact_nav has incorrect column names. Schema was overwritten.

--- Checking Foreign Key Constraints ---
✅ PASSED: Found 1 foreign key links in fact_transactions.


In [1]:
import sqlite3
from pathlib import Path

# 1. Define paths
BASE_DIR = Path.cwd().parent 
SCHEMA_PATH = BASE_DIR / "sql" / "schema.sql"  # Assuming schema.sql is in a 'sql' folder
TEST_DB_PATH = BASE_DIR / "data" / "db" / "test_bluestock.db"

# 2. Connect to the test database (this creates the file if it doesn't exist)
conn = sqlite3.connect(TEST_DB_PATH)

# 3. Read and execute the schema.sql file
with open(SCHEMA_PATH, 'r') as f:
    schema_script = f.read()

try:
    conn.executescript(schema_script)
    print("✅ schema.sql executed without syntax errors.\n")
    
    # 4. Query the internal SQLite master table to verify creation
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
    tables = cursor.fetchall()
    
    print("Tables created in the database:")
    for table in tables:
        print(f" - {table[0]}")
        
except sqlite3.OperationalError as e:
    print(f"❌ SQL Execution Failed: {e}")

finally:
    conn.close()

✅ schema.sql executed without syntax errors.

Tables created in the database:
 - dim_fund
 - dim_date
 - fact_nav
 - fact_transactions
 - fact_performance
 - fact_aum


In [4]:
!python scripts/recommender.py

python: can't open file 'D:\\Programs\\bluestock_mf_capstone\\notebooks\\scripts\\recommender.py': [Errno 2] No such file or directory
